# installs

In [4]:
# import os
# import platform
# import subprocess
# import sys
# !pip install kagglehub

# subprocess.check_call([sys.executable, "-m", "pip", "install", "pip3-autoremove"])
# if platform.system() == "Darwin":
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio"])
# else:
#     subprocess.check_call([
#         sys.executable, "-m", "pip", "install",
#         "torch", "torchvision", "torchaudio", "xformers",
#         "--index-url", "https://download.pytorch.org/whl/cu128",
#     ])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "unsloth"])
# # subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers==4.55.4"])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "trl==0.22.2"])

In [17]:
# %pip install --upgrade "transformers==5.5.0"

# imports

In [2]:
import ctypes
import os
import pandas as pd
import numpy as np
import re
import multiprocessing
from time import time as timer
from tqdm import tqdm
from pathlib import Path
from functools import partial
import requests
import urllib
from PIL import Image

for cuda_driver_path in ("/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib/x86_64-linux-gnu/libcuda.so.1"):
    if Path(cuda_driver_path).exists():
        ctypes.CDLL(cuda_driver_path, mode=ctypes.RTLD_GLOBAL)
        break

from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
from transformers import TextStreamer

from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig


/home/gpuuser7/gpuuser7_a/Divyanshu/GSM8K/.venv/lib/python3.12/site-packages/unsloth/_gpu_init.py:103: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: Detected that PyTorch and TorchAudio were compiled with different CUDA versions. PyTorch has CUDA version 13.0 whereas TorchAudio has CUDA version 12.6. Please install the TorchAudio version that matches your PyTorch version.
  disable_torchaudio_if_cuda_mismatched()
/home/gpuuser7/gpuuser7_a/Divyanshu/GSM8K/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0919 14:38:43.773000 2784324 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0919 14:38:43.811000 2784324 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/home/gpuuser7/gpuuser7_a/Divyanshu/GSM8K/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:3525: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


# download dataest

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("raghavdharwal/amazon-ml-challenge-2025")

print("Path to dataset files:", path)

100%|██████████| 47.7M/47.7M [00:10<00:00, 4.92MB/s]

Extracting files...


Path to dataset files: /home/gpuuser7/gpuuser7_a/.cache/kagglehub/datasets/raghavdharwal/amazon-ml-challenge-2025/versions/1


# extract train subset

In [4]:
# Load only the training data
train_files = list(Path(path).rglob("train.csv"))
if not train_files:
    raise FileNotFoundError(f"train.csv not found under {path}")

train_df = pd.read_csv(train_files[0])
print(f"Train data shape: {train_df.shape}")
display(train_df.head())


Train data shape: (75000, 4)


,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


# img download

In [13]:
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse

IMAGE_FOLDER = (Path.cwd() / "images").resolve()
IMAGE_FOLDER.mkdir(parents=True, exist_ok=True)

def sample_id_key(sample_id):
    value = str(sample_id)
    return value[:-2] if value.endswith(".0") else value

def download_image(image_link, sample_id, image_folder):
    if not isinstance(image_link, str) or not image_link.strip():
        return sample_id, False, "missing image_link"

    url_path = Path(urlparse(image_link).path)
    extension = url_path.suffix.lower() or ".jpg"
    image_path = image_folder / f"{sample_id_key(sample_id)}{extension}"

    if image_path.exists() and image_path.stat().st_size > 0:
        return sample_id, True, "already exists"

    last_error = "download failed"
    for _ in range(3):
        try:
            response = requests.get(
                image_link,
                timeout=30,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()
            if not response.content:
                raise ValueError("empty response")
            image_path.write_bytes(response.content)
            return sample_id, True, "downloaded"
        except Exception as error:
            last_error = str(error)
    return sample_id, False, last_error

download_rows = list(train_df[["sample_id", "image_link"]].itertuples(index=False))
MAX_WORKERS = 64

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    download_results = list(tqdm(
        executor.map(
            lambda row: download_image(row.image_link, row.sample_id, IMAGE_FOLDER),
            download_rows,
        ),
        total=len(download_rows),
    ))

failed_downloads = [result for result in download_results if not result[1]]
print(f"Images available in {IMAGE_FOLDER.resolve()}: {len(download_results) - len(failed_downloads)}")
print(f"Failed downloads: {len(failed_downloads)}")
if failed_downloads:
    display(pd.DataFrame(failed_downloads, columns=["sample_id", "success", "error"]))

image_paths_by_sample = {
    image_path.stem: image_path
    for image_path in IMAGE_FOLDER.iterdir()
    if image_path.is_file() and image_path.stat().st_size > 0
}
print(f"Indexed local image files: {len(image_paths_by_sample)}")

image_available = train_df["sample_id"].map(sample_id_key).isin(image_paths_by_sample)
train_df_with_images = train_df.loc[image_available].copy()
missing_image_rows = train_df.loc[~image_available, ["sample_id", "image_link"]]
print(f"Rows retained with local images: {len(train_df_with_images)} / {len(train_df)}")
if not missing_image_rows.empty:
    print(f"Rows skipped because their image download failed: {len(missing_image_rows)}")
    display(missing_image_rows.head())

100%|██████████| 75000/75000 [00:04<00:00, 15275.39it/s]

Images available in /home/gpuuser7/gpuuser7_a/Divyanshu/GSM8K/finetuningg/images: 74999
Failed downloads: 1


,sample_id,success,error
0,279285,False,404 Client Error: Not Found for url: https://m...


Indexed local image files: 74999
Rows retained with local images: 74999 / 75000
Rows skipped because their image download failed: 1


,sample_id,image_link
38945,279285,https://m.media-amazon.com/images/I/51mjZYDYjy...


# split 

In [14]:
from sklearn.model_selection import train_test_split

# Rebuild the source rows from the image index so stale splits cannot include failed downloads
image_available = train_df["sample_id"].map(sample_id_key).isin(image_paths_by_sample)
train_df_with_images = train_df.loc[image_available].copy()
omitted_rows = train_df.loc[~image_available, ["sample_id", "image_link"]]
print(f"Omitting rows without local images: {len(omitted_rows)}")

train, test = train_test_split(
    train_df_with_images,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

print(f"Train split shape: {train.shape}")
print(f"Test split shape: {test.shape}")

Omitting rows without local images: 1
Train split shape: (59999, 4)
Test split shape: (15000, 4)


# prep

In [15]:
# Prepare multimodal conversations for Qwen2-VL
train_image_available = train["sample_id"].map(sample_id_key).isin(image_paths_by_sample)
test_image_available = test["sample_id"].map(sample_id_key).isin(image_paths_by_sample)
omitted_train_count = int((~train_image_available).sum())
omitted_test_count = int((~test_image_available).sum())
train = train.loc[train_image_available].copy()
test = test.loc[test_image_available].copy()
print(f"Omitted rows without local images before preparation: train={omitted_train_count}, test={omitted_test_count}")

TEXT_COLUMN = next((column for column in ("Content_catalougr", "catalog_content") if column in train.columns), None)
if TEXT_COLUMN is None:
    raise KeyError("Expected Content_catalougr or catalog_content in the dataset")
if "price" not in train.columns or "price" not in test.columns:
    raise KeyError("The train/test splits must contain the price column")

PRICE_INSTRUCTION = """
You are a product pricing assistant. Predict the product price in USD using the catalog text and product image.
Do not treat weights, volumes, quantities, years, or model numbers as the price.
Return only one positive numeric price, without currency symbols or explanation.
"""

def image_path_for_sample(sample_id):
    sample_key = sample_id_key(sample_id)
    if sample_key not in image_paths_by_sample:
        raise FileNotFoundError(f"No downloaded image found for sample_id={sample_id}")
    return image_paths_by_sample[sample_key]

def load_sample_image(sample_id):
    image_path = image_path_for_sample(sample_id)
    with Image.open(image_path) as image:
        return image.convert("RGB")

def convert_row_to_conversation(row, include_answer=True):
    image_path = str(image_path_for_sample(row["sample_id"]))
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": f"{PRICE_INSTRUCTION}\n\n{row[TEXT_COLUMN]}"},
            {"type": "image", "image": image_path},
        ],
    }]
    if include_answer:
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": f"{float(row['price']):.2f}"}],
        })
    return {"messages": messages}

train_dataset = [
    convert_row_to_conversation(row)
    for row in tqdm(train.to_dict(orient="records"), desc="Preparing training examples")
]
print(f"Using text column: {TEXT_COLUMN}")
print(f"Prepared training examples: {len(train_dataset)}")

Omitted rows without local images before preparation: train=0, test=0


Preparing training examples: 100%|██████████| 59999/59999 [00:00<00:00, 69284.86it/s] 

Using text column: catalog_content
Prepared training examples: 59999


# VLM loader + attaching LoRA Adapters

In [ ]:
# Load Qwen2-VL and attach LoRA adapters
if not torch.cuda.is_available():
    raise RuntimeError("Qwen2-VL fine-tuning requires a CUDA GPU runtime")

MODEL_NAME = "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit"
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
USE_GRADIENT_CHECKPOINTING = GPU_MEMORY_GB < 24
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth" if USE_GRADIENT_CHECKPOINTING else False,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print(
    f"Loaded {MODEL_NAME}; GPU memory: {GPU_MEMORY_GB:.1f} GB; "
    f"gradient checkpointing: {USE_GRADIENT_CHECKPOINTING}"
)

==((====))==  Unsloth 2026.9.7: Fast Qwen2_Vl patching. Transformers: 5.5.0. vLLM: 0.28.0+cu129.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.25 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/home/gpuuser7/gpuuser7_a/Divyanshu/GSM8K/.venv/lib/python3.12/site-packages/huggingface_hub/constants.py:301: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Loading weights: 100%|██████████| 730/730 [00:01<00:00, 704.33it/s] 


Loaded unsloth/Qwen2-VL-7B-Instruct-bnb-4bit with LoRA adapters


# Fine-tuning

In [20]:
# Fine-tune on the 80% training split
MAX_SEQ_LENGTH = 768
GPU_MEMORY_GB = 80
EFFECTIVE_BATCH_SIZE = 8
if GPU_MEMORY_GB >= 48:
    PER_DEVICE_TRAIN_BATCH_SIZE = 8
elif GPU_MEMORY_GB >= 24:
    PER_DEVICE_TRAIN_BATCH_SIZE = 4
elif GPU_MEMORY_GB >= 12:
    PER_DEVICE_TRAIN_BATCH_SIZE = 2
else:
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = max(1, EFFECTIVE_BATCH_SIZE // PER_DEVICE_TRAIN_BATCH_SIZE)
DATALOADER_NUM_WORKERS = 2
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_TF32 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    data_collator=UnslothVisionDataCollator(
        model,
        tokenizer,
        max_seq_length=MAX_SEQ_LENGTH,
    ),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="qwen2vl_price_training",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQ_LENGTH,
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        bf16=USE_BF16,
        fp16=not USE_BF16,
        tf32=USE_TF32,
    ),
)
print(
    f"GPU memory: {GPU_MEMORY_GB:.1f} GB; "
    f"micro-batch: {PER_DEVICE_TRAIN_BATCH_SIZE}; "
    f"gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}; "
    f"effective batch: {EFFECTIVE_BATCH_SIZE}"
)
trainer_stats = trainer.train()
print(trainer_stats.metrics)

Unsloth: Model does not have a default image size - using 512
GPU memory: 80.0 GB; micro-batch: 8; gradient accumulation: 1; effective batch: 8


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 59,999 | Num Epochs = 1 | Total steps = 7,500
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 50,855,936 of 8,342,231,552 (0.61% trained)


ValueError: Caught ValueError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/gpuuser7/gpuuser7_a/Divyanshu/GSM8K/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 374, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/gpuuser7/gpuuser7_a/Divyanshu/GSM8K/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/gpuuser7/gpuuser7_a/Divyanshu/GSM8K/.venv/lib/python3.12/site-packages/unsloth/trainer.py", line 85, in __call__
    return super().__call__(examples)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/gpuuser7/gpuuser7_a/Divyanshu/GSM8K/.venv/lib/python3.12/site-packages/unsloth_zoo/vision_utils.py", line 1465, in __call__
    batch = self.processor(**proc_kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/gpuu

# Testing

In [ ]:
# Generate predictions for the 20% test split and calculate SMAPE
FastVisionModel.for_inference(model)

def extract_price(text):
    match = re.search(r"(?<!\d)(\d+(?:\.\d+)?)(?!\d)", text.replace(",", ""))
    if match is None:
        return np.nan
    value = float(match.group(1))
    return value if np.isfinite(value) and value > 0 else np.nan

def predict_price(row):
    image = load_sample_image(row["sample_id"])
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": f"{PRICE_INSTRUCTION}\n\n{row[TEXT_COLUMN]}"},
            {"type": "image", "image": image},
        ],
    }]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        images=image,
        text=input_text,
        add_special_tokens=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            use_cache=True,
        )
    prompt_length = inputs["input_ids"].shape[-1]
    generated_text = tokenizer.decode(
        output_ids[0][prompt_length:],
        skip_special_tokens=True,
    ).strip()
    return generated_text, extract_price(generated_text)

raw_predictions = []
predicted_prices = []
for row in tqdm(test.to_dict(orient="records"), desc="Evaluating test split"):
    raw_prediction, predicted_price = predict_price(row)
    raw_predictions.append(raw_prediction)
    predicted_prices.append(predicted_price)

test_results = test[["sample_id", "price"]].copy()
test_results["raw_prediction"] = raw_predictions
test_results["predicted_price"] = predicted_prices
fallback_price = float(train["price"].median())
test_results["predicted_price"] = test_results["predicted_price"].fillna(fallback_price).clip(lower=0.01)

def smape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    denominator = np.abs(actual) + np.abs(predicted)
    values = np.where(denominator == 0, 0.0, 2.0 * np.abs(predicted - actual) / denominator)
    return 100.0 * values.mean()

invalid_count = pd.isna(predicted_prices).sum()
score = smape(test_results["price"], test_results["predicted_price"])
print(f"Invalid generated prices replaced with median: {invalid_count}")
print(f"Test SMAPE: {score:.4f}%")
display(test_results.head())

OUTPUT_DIR = Path("qwen2vl_price_lora")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
test_results.to_csv("qwen2vl_test_predictions.csv", index=False)
print(f"Saved LoRA adapter and tokenizer to {OUTPUT_DIR.resolve()}")
print("Saved predictions to qwen2vl_test_predictions.csv")
# vLLM is optional for faster large-scale inference; Transformers generation above is sufficient for this evaluation.